In [6]:
import numpy as np
import pandas as pd
import os

# Load raw subscription data (Bronze)

In [3]:
!git clone https://github.com/vinculum3141-ship-it/utilus_home_assignment.git

Cloning into 'utilus_home_assignment'...
remote: Enumerating objects: 88, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 88 (delta 6), reused 86 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (88/88), 132.05 KiB | 2.45 MiB/s, done.
Resolving deltas: 100% (6/6), done.


In [7]:
base_path = '/content/utilus_home_assignment/assignment/'

In [67]:
subs_raw = pd.read_csv(os.path.join(base_path, "subscriptions.csv"))
print("Subscription data loaded successfully. All rows are kept.")
print(f"Total rows in subs_raw: {len(subs_raw)}")
print(f"Number of unique customer_ids: {subs_raw['customer_id'].nunique()}")
print("Subscriptions data loaded successfully. First 5 rows:")
display(subs_raw.head())

Subscription data loaded successfully. All rows are kept.
Total rows in subs_raw: 54
Number of unique customer_ids: 41
Subscriptions data loaded successfully. First 5 rows:


,customer_id,start_date,end_date,plan,monthly_price
0,C001,2024-01-02,2024-03-31,basic,30
1,C001,2024-04-01,NaN,pro,50
2,C002,2024-01-11,2024-02-10,basic,25
3,C002,2024-02-25,NaN,basic,25
4,C003,2024-01-27,2024-02-28,baisc,30


# Clean and validate subscription data (Silver)

In [78]:
subs_silver = subs_raw.copy()
print(f"Silver DataFrame 'subs_silver' created with {len(subs_silver)} rows.")
display(subs_silver.head())

Silver DataFrame 'subs_silver' created with 54 rows.


,customer_id,start_date,end_date,plan,monthly_price
0,C001,2024-01-02,2024-03-31,basic,30
1,C001,2024-04-01,NaN,pro,50
2,C002,2024-01-11,2024-02-10,basic,25
3,C002,2024-02-25,NaN,basic,25
4,C003,2024-01-27,2024-02-28,baisc,30


## Inspect customer_id field

In [79]:
subs_silver['customer_id'] = subs_silver['customer_id'].str.strip()
print("Trailing whitespaces removed from 'customer_id' column.")

Trailing whitespaces removed from 'customer_id' column.


In [80]:
# Verify no empty customer ID fields
empty_customer_id_mask = subs_silver['customer_id'].replace('', np.nan).str.strip().isna()
empty_customer_id_entries = subs_silver[empty_customer_id_mask]
if not empty_customer_id_entries.empty:
    print(f"\nWARNING: Found {len(empty_customer_id_entries) } empty customer IDs, correct data at source")
    display(empty_customer_id_entries[['customer_id', 'start_date']])
else:
    print("No entries found with empty 'customer_id'.")

assert empty_customer_id_entries.empty, "Cannot process empty customer_id field"

No entries found with empty 'customer_id'.


## Inspect start_date field

In [81]:
subs_silver['start_date'] = subs_silver['start_date'].str.strip()
print("Trailing whitespaces removed from 'start_date' column.")

Trailing whitespaces removed from 'start_date' column.


In [83]:
# Identify entries with empty start_date
empty_start_date_mask = subs_silver['start_date'].replace('', np.nan).str.strip().isna()
empty_start_date_entries = subs_silver[empty_start_date_mask]

if not empty_start_date_entries.empty:
    print("WARNING: Found entries with empty 'start_date':")
    for index, row in empty_start_date_entries.iterrows():
        print(f"Subscription entry {row['customer_id']} has empty start date, correct data at source")
else:
    print("No entries found with empty 'start_date'.")

assert empty_start_date_entries.empty, "Cannot infer analysis, start date must exist"

No entries found with empty 'start_date'.


In [85]:
# Attempt to convert 'start_date' to datetime and identify malformed entries
malformed_start_date_mask = pd.to_datetime(subs_silver['start_date'], errors='coerce').isna()

# Identify rows where temporary datetime column is NaT
malformed_start_dates = subs_silver[malformed_start_date_mask]
malformed_start_date_count = len(malformed_start_dates)

if malformed_start_date_count > 0:
    print(f"\nWARNING: Found {malformed_start_date_count} malformed 'start_date' strings that could not be parsed")
    display(malformed_start_dates[['customer_id', 'start_date']])

    for index, row in malformed_start_dates.iterrows():
      print(f"Subscription entry {row['customer_id']} has malformed start date, correct data at source")

else:
    print("No malformed 'start_date' entries found.")

assert malformed_start_dates.empty, "Cannot infer analysis, start date must exist"

No malformed 'start_date' entries found.


## Inspect end_date field

Note: empty entries allowed, assume ongoing subscription if no end_date

In [86]:
subs_silver['end_date'] = subs_silver['end_date'].str.strip()
print("Trailing whitespaces removed from 'end_date' column.")

Trailing whitespaces removed from 'end_date' column.


In [88]:
# 1. Identify and count genuinely empty values in subs_silver['end_date']
# First, normalize empty strings to NaN for consistent counting
subs_silver['end_date_clean'] = subs_silver['end_date'].replace('', np.nan).str.strip()
genuinely_empty_count = subs_silver['end_date_clean'].isna().sum()

# 2. Create a temporary column by attempting to convert subs_silver['end_date'] to datetime
temp_datetime_end_date = pd.to_datetime(subs_silver['end_date'], errors='coerce')

# 3. Identify rows where temporary datetime column is NaT, but original subs_silver['end_date_clean'] was *not* genuinely empty
# These are malformed strings that were not empty but couldn't be parsed
malformed_dates = subs_silver[
    temp_datetime_end_date.isna() &
    subs_silver['end_date_clean'].notna()
]
malformed_count = len(malformed_dates)

print(f"Genuinely empty 'end_date' values (NaN, None, empty string): {genuinely_empty_count}")
print(f"Malformed 'end_date' string values (could not be parsed but were not empty): {malformed_count}")

if malformed_count > 0:
    print("\nExamples of malformed 'end_date' strings (original and coerced):")
    display(malformed_dates[['customer_id', 'end_date']]) # Corrected line

# Drop the temporary column
subs_silver = subs_silver.drop(columns=['end_date_clean'])

Genuinely empty 'end_date' values (NaN, None, empty string): 36
Malformed 'end_date' string values (could not be parsed but were not empty): 1

Examples of malformed 'end_date' strings (original and coerced):


,customer_id,end_date
7,C005,2024-02-30


## Inspect plan field

In [89]:
subs_silver['plan'] = subs_silver['plan'].str.strip()
print("Trailing whitespaces removed from 'plan' column.")

print(f"Number of unique plans: {subs_silver['plan'].nunique()}")
print(f"Unique plans: {subs_silver['plan'].unique()}")

Trailing whitespaces removed from 'plan' column.
Number of unique plans: 3
Unique plans: ['basic' 'pro' 'baisc']


In [90]:
subs_silver['plan'] = subs_silver['plan'].replace('baisc', 'basic')
print("Corrected 'baisc' to 'basic' in 'plan' column.")

print(f"Number of unique plans after correction: {subs_silver['plan'].nunique()}")
print(f"Unique plans after correction: {subs_silver['plan'].unique()}")

Corrected 'baisc' to 'basic' in 'plan' column.
Number of unique plans after correction: 2
Unique plans after correction: ['basic' 'pro']


## Inspect monthly_price field

In [93]:
subs_silver['monthly_price'] = subs_silver['monthly_price'].str.strip()
print("Trailing whitespaces removed from 'monthly_price' column.")

Trailing whitespaces removed from 'monthly_price' column.


In [95]:
# Attempt to convert 'monthly_price' to numeric and identify non-numeric entries
subs_silver['monthly_price_numeric'] = pd.to_numeric(subs_silver['monthly_price'], errors='coerce')

# Identify rows where the numeric conversion resulted in NaN (meaning original was not numeric)
non_numeric_monthly_price_mask = subs_silver['monthly_price_numeric'].isna()
non_numeric_monthly_price_entries = subs_silver[non_numeric_monthly_price_mask]

if not non_numeric_monthly_price_entries.empty:
    print(f"WARNING: Found {len(non_numeric_monthly_price_entries)} non-numeric 'monthly_price' entries:")
    display(non_numeric_monthly_price_entries[['customer_id', 'monthly_price']])

    for index, row in non_numeric_monthly_price_entries.iterrows():
        print(f"Subscription entry {row['customer_id']} has non-numeric monthly_price '{row['monthly_price']}'. Correct data at source.")
else:
    print("All 'monthly_price' entries are numeric.")

# Drop the temporary numeric column
subs_silver = subs_silver.drop(columns=['monthly_price_numeric'])

assert non_numeric_monthly_price_entries.empty, "Non-numeric 'monthly_price' values found. Please correct."

,customer_id,monthly_price
29,C021,thirty


Subscription entry C021 has non-numeric monthly_price 'thirty'. Correct data at source.


AssertionError: Non-numeric 'monthly_price' values found. Please correct.

In [96]:
# Replace 'thirty' with '30' in the 'monthly_price' column
subs_silver['monthly_price'] = subs_silver['monthly_price'].replace('thirty', '30')
print("Replaced 'thirty' with '30' in 'monthly_price' column.")

# Attempt to convert 'monthly_price' to numeric and identify non-numeric entries
subs_silver['monthly_price_numeric'] = pd.to_numeric(subs_silver['monthly_price'], errors='coerce')

# Identify rows where the numeric conversion resulted in NaN (meaning original was not numeric)
non_numeric_monthly_price_mask = subs_silver['monthly_price_numeric'].isna()
non_numeric_monthly_price_entries = subs_silver[non_numeric_monthly_price_mask]

if not non_numeric_monthly_price_entries.empty:
    print(f"WARNING: Found {len(non_numeric_monthly_price_entries)} non-numeric 'monthly_price' entries:")
    display(non_numeric_monthly_price_entries[['customer_id', 'monthly_price']])
else:
    print("All 'monthly_price' entries are numeric after correction.")

# Drop the temporary numeric column
subs_silver = subs_silver.drop(columns=['monthly_price_numeric'])

assert non_numeric_monthly_price_entries.empty, "Non-numeric 'monthly_price' values still found after correction. Please investigate."

Replaced 'thirty' with '30' in 'monthly_price' column.
All 'monthly_price' entries are numeric after correction.


# View clean data

In [97]:
display(subs_silver)

,customer_id,start_date,end_date,plan,monthly_price
0,C001,2024-01-02,2024-03-31,basic,30
1,C001,2024-04-01,NaN,pro,50
2,C002,2024-01-11,2024-02-10,basic,25
3,C002,2024-02-25,NaN,basic,25
4,C003,2024-01-27,2024-02-28,basic,30
5,C004,2024-02-05,2024-04-01,pro,55
6,C004,2024-03-15,2024-05-01,pro,55
7,C005,2024-02-28,2024-02-30,basic,20
8,C006,2024-03-03,2024-03-20,basic,20
9,C006,2024-04-01,NaN,basic,20


In [98]:
output_path = '/content/utilus_home_assignment/output/'

# Ensure the output directory exists
if not os.path.exists(output_path):
    os.makedirs(output_path)

# Save the cleaned DataFrame to a CSV file
output_file = os.path.join(output_path, 'subscriptions_silver.csv')
subs_silver.to_csv(output_file, index=False)

print(f"Cleaned data saved to {output_file}")

Cleaned data saved to /content/utilus_home_assignment/output/subscriptions_silver.csv
